<a href="https://colab.research.google.com/github/Foyceek/MLF_2026_HECL_Frantisek/blob/main/MLF_Project_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Imports

In [ ]:
!pip install optuna -q

import os
import shutil
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import optuna
from optuna.importance import get_param_importances
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

from google.colab import drive

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.8 MB/s eta 0:00:00


#  Google Drive Mounting

In [ ]:
# remove mounted drive directory if it's not responding
# shutil.rmtree('/content/drive')

# mount google drive
drive.mount('/content/drive', force_remount=True)
# list it's contents
# !ls /content/drive/MyDrive/MLF/datasets

Mounted at /content/drive
x_test.zip  x_train.zip  y_test_submission_example_v2.csv  y_train_v2.csv


#Dataset paths

In [ ]:
DRIVE_DATASET_PATH = "/content/drive/MyDrive/MLF/datasets"

# Define paths for zipped training and testing data, and the CSV labels file in Google Drive
TRAIN_ZIP = os.path.join(DRIVE_DATASET_PATH, "x_train.zip")
TEST_ZIP = os.path.join(DRIVE_DATASET_PATH, "x_test.zip")
CSV_DRIVE_PATH = os.path.join(DRIVE_DATASET_PATH, "y_train_v2.csv")

# Define a local path for datasets for faster access during training
LOCAL_DATASET_PATH = "/content/datasets"

# Define local directories where the zipped data will be extracted
TRAIN_DIR = os.path.join(LOCAL_DATASET_PATH, "x_train")
TEST_DIR = os.path.join(LOCAL_DATASET_PATH, "x_test")

# Define the local path for the labels CSV file
LABELS_FILE = os.path.join(LOCAL_DATASET_PATH, "y_train_v2.csv")

# Create the local dataset directory if it doesn't already exist
os.makedirs(LOCAL_DATASET_PATH, exist_ok=True)

# Define a directory for saving Optuna run results and models in Google Drive
SAVE_DIR = "/content/drive/MyDrive/MLF/optuna_runs/"
os.makedirs(SAVE_DIR, exist_ok=True)

# Define a directory for saving plots in Google Drive
PLOT_DIR = "/content/drive/MyDrive/MLF/plots/"
os.makedirs(PLOT_DIR, exist_ok=True)

def extract_if_needed(zip_path, extract_to):
    """Extracts a zip file to a specified directory if the directory doesn't already exist."""
    if os.path.exists(extract_to):
        print(f"{extract_to} already exists, skipping extraction.")
        return

    os.makedirs(extract_to, exist_ok=True)

    print(f"Extracting {zip_path} to {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

    print("Extraction finished.")

# Extract each ZIP into its own directory for training and testing data
extract_if_needed(TRAIN_ZIP, TRAIN_DIR)
extract_if_needed(TEST_ZIP, TEST_DIR)

# Copy the labels CSV file locally for faster access during model training
if not os.path.exists(LABELS_FILE):
    shutil.copy(CSV_DRIVE_PATH, LABELS_FILE)
    print("CSV copied to local dataset folder.")
else:
    print("CSV already exists locally.")


Extracting /content/drive/MyDrive/MLF/datasets/x_train.zip to /content/datasets/x_train...
Extraction finished.
Extracting /content/drive/MyDrive/MLF/datasets/x_test.zip to /content/datasets/x_test...
Extraction finished.
CSV copied to local dataset folder.


#Dataset classes

In [ ]:
class DopplerDataset(Dataset):
    """Custom Dataset for training and validation data."""
    def __init__(self, df, folder, transform=None):

        self.folder = folder
        self.transform = transform

        # Adjust image IDs (add 1) and convert to integer numpy array
        self.ids = (df["id"].to_numpy() + 1).astype(int)
        # Convert target labels to integer numpy array
        self.labels = df["target"].to_numpy().astype(int)

        # Create a list of full image paths
        self.paths = [
            os.path.join(folder, f"img_{img_id}.png")
            for img_id in self.ids
        ]

    def __len__(self):
        """Returns the total number of samples in the dataset."""
        return len(self.labels)

    def __getitem__(self, idx):
        """Retrieves the image and its corresponding label at the given index."""
        # Open image from path and convert to RGB
        image = Image.open(self.paths[idx]).convert("RGB")

        # Apply transformations if any are defined
        if self.transform:
            image = self.transform(image)

        # Return the transformed image and its label as a PyTorch tensor
        return image, torch.tensor(self.labels[idx], dtype=torch.long)


class TestDataset(Dataset):
    """Custom Dataset for test data, without labels."""
    def __init__(self, folder, transform=None):

        self.folder = folder
        self.transform = transform

        # List and sort image files based on their numerical ID
        self.files = sorted(
            os.listdir(folder),
            key=lambda x: int(x.split("_")[1].split(".")[0])
        )

    def __len__(self):
        """Returns the total number of samples in the dataset."""
        return len(self.files)

    def __getitem__(self, idx):
        """Retrieves the image and its original ID at the given index."""
        file = self.files[idx]
        path = os.path.join(self.folder, file)

        # Open image from path and convert to RGB
        image = Image.open(path).convert("RGB")

        # Apply transformations if any are defined
        if self.transform:
            image = self.transform(image)

        # Extract image ID from the filename (e.g., 'img_123.png' -> 123)
        img_id = int(file.split("_")[1].split(".")[0])

        # Return the transformed image and its original ID
        return image, img_id

#Training

In [ ]:
# =========================
# CONFIGURATION
# =========================

BATCH_SIZE = 32
EPOCHS = 100
LR = 0.0005
IMAGE_SIZE = 224

PATIENCE = 30 # Early stopping patience for final training
ID_SHIFT = 1 # Shift for image IDs (if needed, e.g., 0-indexed vs 1-indexed filenames)
NUM_WORKERS = 2 # Number of worker processes for data loading

N_TRIALS = 30  # Number of trials for Optuna hyperparameter tuning (increase for better results)

# Toggle for hyperparameter tuning: Set to True to run Optuna, False to use pre-defined best parameters
RUN_HYPERPARAM_TUNING = False

print(f"RUN_HYPERPARAM_TUNING: {RUN_HYPERPARAM_TUNING}")

# Determine the device for training (GPU if available, otherwise CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Initialize GradScaler for mixed precision training if using CUDA
scaler = torch.amp.GradScaler(device="cuda", enabled=(DEVICE.type == "cuda"))

print("Using device:", DEVICE)

# Default best parameters for warm start in Optuna or when skipping Optuna
DEFAULT_BEST_PARAMS = {
    # These parameters were found from previous Optuna runs and are used for faster experimentation
    'lr': 0.0009315891190192857,
    'batch_size': 32,
    'weight_decay': 4.3898184826770925e-06,
    'img_size': 160,
    'dropout': 0.11350523833817275
}

# =========================
# LOGGING UTILITY
# =========================
def log(msg: str):
    """Prints a timestamped message to the console."""
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

# =========================
# GLOBAL TRACKING VARIABLES (for Optuna study)
# =========================
trial_history = [] # Stores results of each Optuna trial
best_global_acc = 0.0 # Tracks the best accuracy found across all Optuna trials
best_global_trial = None # Stores the trial number that achieved the best accuracy

# =========================
# OPTUNA PRUNER CONFIGURATION
# =========================
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=5, # Number of trials to run before starting pruning
    n_warmup_steps=2 # Number of epochs to wait in each trial before starting pruning
)

# =========================
# OPTUNA OBJECTIVE FUNCTION
# =========================
def objective(trial):
    """Defines the objective function for Optuna to optimize.

    Args:
        trial (optuna.trial.Trial): A trial object from Optuna.

    Returns:
        float: The validation accuracy achieved by the model during the trial.
    """

    global best_global_acc, best_global_trial

    # Hyperparameter suggestions for Optuna
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    img_size = trial.suggest_categorical("img_size", [160, 192, 224])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)

    log(f"Trial {trial.number} START")
    log(f"lr={lr:.6f}, batch={batch_size}, wd={weight_decay:.6f}, img={img_size}, dropout={dropout:.2f}")

    # Image transformations for training data (with data augmentation)
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
    ])

    # Image transformations for validation data (no data augmentation)
    val_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
    ])

    # Load labels and split into training and validation sets
    df = pd.read_csv(LABELS_FILE)
    train_df, val_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df["target"],
        random_state=42
    )

    # Create datasets and data loaders
    train_ds = DopplerDataset(train_df, TRAIN_DIR, train_tf)
    val_ds = DopplerDataset(val_df, TRAIN_DIR, val_tf)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    # Load a pre-trained ResNet-18 model and freeze its weights
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    for p in model.parameters():
        p.requires_grad = False

    # Replace the final classification layer with a new one for 4 classes
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(model.fc.in_features, 4)
    )

    # Move model to the specified device (GPU/CPU)
    model = model.to(DEVICE)

    # Define loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.fc.parameters(), # Only optimize the new classification layer
        lr=lr,
        weight_decay=weight_decay
    )
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

    best_acc = 0 # Tracks the best accuracy for the current trial
    counter = 0 # Counter for early stopping within a trial

    # Training loop for the current trial
    for epoch in range(EPOCHS):

        model.train() # Set model to training mode

        # Iterate over training data
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad() # Zero gradients

            # Forward pass with mixed precision
            with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
                out = model(x)
                loss = criterion(out, y)

            # Backward pass and optimization step
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        scheduler.step() # Update learning rate scheduler

        model.eval() # Set model to evaluation mode
        correct, total = 0, 0

        # Evaluate on validation set
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)

                out = model(x)
                preds = torch.argmax(out, dim=1) # Get predicted class

                correct += (preds == y).sum().item()
                total += y.size(0)

        acc = correct / total # Calculate accuracy

        log(f"Trial {trial.number} | Epoch {epoch+1} | acc={acc:.4f}")

        trial.report(acc, epoch) # Report current accuracy to Optuna

        # Pruning check
        if trial.should_prune():
            log(f"Trial {trial.number} PRUNED")
            raise optuna.exceptions.TrialPruned()

        # Early stopping logic for the current trial
        if acc > best_acc:
            best_acc = acc
            counter = 0
        else:
            counter += 1

        if counter >= PATIENCE:
            break # Stop training if accuracy hasn't improved for PATIENCE epochs

    # Store trial results
    trial_history.append({
        "trial": trial.number,
        "lr": lr,
        "batch_size": batch_size,
        "weight_decay": weight_decay,
        "img_size": img_size,
        "dropout": dropout,
        "accuracy": best_acc
    })

    # Update global best accuracy if current trial is better
    if best_acc > best_global_acc:
        best_global_acc = best_acc
        best_global_trial = trial.number

    log(f"Trial {trial.number} DONE | best={best_acc:.4f}")

    return best_acc


# =========================
# HYPERPARAMETER TUNING EXECUTION
# =========================
if RUN_HYPERPARAM_TUNING:
    log("Starting Optuna study")

    # Create or load an Optuna study
    study = optuna.create_study(
        direction="maximize", # Maximize validation accuracy
        pruner=pruner, # Use the defined pruner
        storage=f"sqlite:///{SAVE_DIR}study.db", # Store study results in a SQLite database
        load_if_exists=True # Load existing study if available
    )

    # Warm start the study with default best parameters if it's new or empty
    if not study.trials:
        log("Enqueuing warm-start trial.")
        study.enqueue_trial(DEFAULT_BEST_PARAMS)

    # Run the optimization for N_TRIALS
    study.optimize(objective, n_trials=N_TRIALS)
    log("Optuna finished")

    print("\nBest params:", study.best_params)
    print("Best accuracy:", study.best_value)

    # =========================
    # SAVE OPTUNA HISTORY AND PLOTS
    # =========================

    # Save optimization history to CSV
    df_hist = pd.DataFrame(trial_history)
    df_hist.to_csv(SAVE_DIR + "history.csv", index=False)

    # ---- Optimization history plot ----
    plt.figure()
    plt.plot(df_hist["trial"], df_hist["accuracy"], marker="o")
    best_so_far = df_hist["accuracy"].cummax()
    plt.plot(df_hist["trial"], best_so_far, linestyle="--")
    plt.title("Optimization History")
    plt.xlabel("Trial")
    plt.ylabel("Accuracy")
    plt.grid(True)
    plt.savefig(PLOT_DIR + "optuna_history.png", dpi=300)
    plt.close()
    log(f"Plot saved: {PLOT_DIR}optuna_history.png")

    # ---- Parallel coordinates plot ----
    ax = optuna.visualization.matplotlib.plot_parallel_coordinate(study)
    ax.figure.set_size_inches(18, 9)
    for coll in ax.collections:
        coll.set_cmap("viridis") # Apply viridis colormap
    ax.tick_params(axis="both", which="major", labelsize=9)
    for label in ax.get_xticklabels():
        label.set_rotation(30)
        label.set_horizontalalignment("right")
    ax.set_title("Parallel Coordinate Plot (Optuna)", fontsize=14)
    ax.figure.tight_layout()
    ax.figure.savefig(PLOT_DIR + "optuna_parallel.png", dpi=300, bbox_inches="tight")
    plt.close(ax.figure)
    log(f"Plot saved: {PLOT_DIR}optuna_parallel.png")

    # ---- Parameter importance plot ----
    importances = get_param_importances(study)
    plt.figure()
    plt.bar(importances.keys(), importances.values())
    plt.xticks(rotation=45)
    plt.title("Parameter Importances")
    plt.tight_layout()
    plt.savefig(PLOT_DIR + "optuna_importance.png", dpi=300)
    plt.close()
    log(f"Plot saved: {PLOT_DIR}optuna_importance.png")

    # ---- Custom accuracy progress plot ----
    plt.figure()
    plt.plot(df_hist["trial"], df_hist["accuracy"], marker="o")
    plt.title("Optuna Accuracy Progress (custom)")
    plt.xlabel("Trial")
    plt.ylabel("Accuracy")
    plt.grid(True)
    plt.savefig(PLOT_DIR + "optuna_progress_custom.png", dpi=300)
    plt.close()
    log(f"Plot saved: {PLOT_DIR}optuna_progress_custom.png")

    # ---- Hyperparameter scatter matrix ----
    fig, axs = plt.subplots(2, 3, figsize=(16, 8))
    axs[0, 0].scatter(df_hist["lr"], df_hist["accuracy"])
    axs[0, 0].set_xscale("log")
    axs[0, 0].set_title("LR vs Accuracy")
    axs[0, 1].scatter(df_hist["weight_decay"], df_hist["accuracy"])
    axs[0, 1].set_xscale("log")
    axs[0, 1].set_title("Weight Decay vs Accuracy")
    axs[0, 2].scatter(df_hist["img_size"], df_hist["accuracy"])
    axs[0, 2].set_title("Image Size vs Accuracy")
    axs[1, 0].scatter(df_hist["batch_size"], df_hist["accuracy"])
    axs[1, 0].set_title("Batch Size vs Accuracy")
    axs[1, 1].scatter(df_hist["dropout"], df_hist["accuracy"])
    axs[1, 1].set_title("Dropout vs Accuracy")
    axs[1, 2].axis("off") # Hide unused subplot
    plt.tight_layout()
    plt.savefig(PLOT_DIR + "hyperparams_analysis.png", dpi=300)
    plt.close()
    log(f"Plot saved: {PLOT_DIR}hyperparams_analysis.png")

    best = study.best_params # Get the best parameters from the Optuna study

else:
    log("Skipping Optuna. Using pre-defined best parameters.")
    best = DEFAULT_BEST_PARAMS # Use default parameters if Optuna is skipped

# =========================
# FINAL MODEL TRAINING
# =========================

# Unpack best parameters for final training
lr = best["lr"]
batch_size = best["batch_size"]
img_size = best["img_size"]
weight_decay = best["weight_decay"]
dropout = best["dropout"]

log(f"Final training with best params: {best}")

# Image transformations for training (with data augmentation)
train_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

# Image transformations for validation/testing (no data augmentation)
val_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

# Load labels and split into training and validation sets
df = pd.read_csv(LABELS_FILE)
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["target"],
    random_state=42
)

# Create datasets and data loaders for final training
train_ds = DopplerDataset(train_df, TRAIN_DIR, train_tf)
val_ds = DopplerDataset(val_df, TRAIN_DIR, val_tf)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Initialize ResNet-18 model and replace its final classification layer
model = resnet18(weights=ResNet18_Weights.DEFAULT)
# Note: For final training, we train the entire model, not just the head.
model.fc = nn.Sequential(
    nn.Dropout(dropout),
    nn.Linear(model.fc.in_features, 4)
)
model = model.to(DEVICE)

# Define loss function, optimizer, and scheduler
criterion = nn.CrossEntropyLoss()
# Optimize all model parameters for final training
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# Early stopping variables for final training run
best_val_accuracy = 0.0
patience_counter = 0
best_model_state_path = os.path.join(SAVE_DIR, "best_final_model.pth")

# Final training loop
for epoch in range(EPOCHS):

    model.train() # Set model to training mode

    # Iterate over training data
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad() # Zero gradients

        # Forward pass with mixed precision
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            out = model(x)
            loss = criterion(out, y)

        # Backward pass and optimization step
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    scheduler.step() # Update learning rate

    model.eval() # Set model to evaluation mode
    correct, total = 0, 0

    # Evaluate on validation set to check performance
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = torch.argmax(model(x), dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    current_val_accuracy = correct / total
    log(f"Final epoch {epoch+1} | acc={current_val_accuracy:.4f}")

    # Early stopping logic for the final training
    if current_val_accuracy > best_val_accuracy:
        best_val_accuracy = current_val_accuracy
        patience_counter = 0
        torch.save(model.state_dict(), best_model_state_path) # Save the best model
        log(f"New best model saved with accuracy: {best_val_accuracy:.4f}")
    else:
        # Start checking for early stopping only after a certain number of epochs (e.g., 30)
        if (epoch + 1) >= 30:
            patience_counter += 1
            log(f"Validation accuracy did not improve. Patience: {patience_counter}/{PATIENCE}")
            if patience_counter >= PATIENCE:
                log(f"Early stopping triggered after {epoch + 1} epochs.")
                break
        else:
            log(f"Validation accuracy did not improve. Early stopping not active yet (epoch {epoch+1}/30).")

log("Final training done")

# Load the state of the best model saved during final training
if os.path.exists(best_model_state_path):
    model.load_state_dict(torch.load(best_model_state_path))
    log("Best model state loaded for final evaluation and submission.")
else:
    # This case should ideally not happen if training ran and saved at least one best model.
    log("No best model state found. Proceeding with the last trained model state.")

# Prepare for final validation evaluation metrics and confusion matrix
val_preds_final = []
val_labels_final = []

model.eval() # Ensure model is in evaluation mode
with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        preds = torch.argmax(model(x), dim=1)
        val_preds_final.extend(preds.cpu().numpy())
        val_labels_final.extend(y.cpu().numpy())

# =========================
# PLOTS AFTER FINAL TRAINING
# =========================

# Confusion Matrix plot
cm = confusion_matrix(val_labels_final, val_preds_final)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap='viridis') # Use viridis colormap for better visibility
plt.title('Confusion Matrix')
plt.colorbar()
classes = [str(i) for i in range(4)] # Assuming 4 classes: 0, 1, 2, 3
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes, rotation=45)
plt.yticks(tick_marks, classes)

# Add values to the confusion matrix cells for readability
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black") # Adjust text color based on background

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig(PLOT_DIR + "confusion_matrix.png", dpi=300)
plt.close()
log(f"Plot saved: {PLOT_DIR}confusion_matrix.png")


# =========================
# GENERATE SUBMISSION FILE
# =========================

# Prepare test dataset and loader
test_dataset = TestDataset(TEST_DIR, val_tf)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)

model.eval() # Set model to evaluation mode

ids, preds = [], []

# Make predictions on the test set
with torch.no_grad():
    for x, img_ids in test_loader:
        x = x.to(DEVICE)
        pred = torch.argmax(model(x), dim=1).cpu().numpy()
        ids.extend((img_ids.numpy() - ID_SHIFT).tolist()) # Adjust IDs back if a shift was applied
        preds.extend(pred.tolist())

# Create submission DataFrame and save to CSV
submission = pd.DataFrame({"id": ids, "target": preds})
submission = submission.sort_values("id") # Ensure submission is sorted by ID

submission.to_csv(SAVE_DIR + "submission.csv", index=False)

log("Submission saved to Drive")

RUN_HYPERPARAM_TUNING: False
Using device: cuda
[19:42:30] Skipping Optuna. Using pre-defined best parameters.
[19:42:30] Final training with best params: {'lr': 0.0009315891190192857, 'batch_size': 32, 'weight_decay': 4.3898184826770925e-06, 'img_size': 160, 'dropout': 0.11350523833817275}
[19:43:04] Final epoch 1 | acc=0.7866
[19:43:04] New best model saved with accuracy: 0.7866
[19:43:19] Final epoch 2 | acc=0.8380
[19:43:19] New best model saved with accuracy: 0.8380
[19:43:34] Final epoch 3 | acc=0.9041
[19:43:34] New best model saved with accuracy: 0.9041
[19:43:49] Final epoch 4 | acc=0.9198
[19:43:50] New best model saved with accuracy: 0.9198
[19:44:06] Final epoch 5 | acc=0.9177
[19:44:06] Validation accuracy did not improve. Early stopping not active yet (epoch 5/30).
[19:44:20] Final epoch 6 | acc=0.9410
[19:44:21] New best model saved with accuracy: 0.9410
[19:44:36] Final epoch 7 | acc=0.9512
[19:44:36] New best model saved with accuracy: 0.9512
[19:44:51] Final epoch 8 |